# Homework 1: Can This Support Retriever Be Trusted?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module4/week4_homework_retrieval.ipynb)

## Scenario

You are an **NLP engineer** helping a small product team build a retrieval app for the fictional **Northstar University Technology Help Center**. The app matches a user's short question to useful support articles. Your job is to test the retriever and recommend one operating mode: **automatic top-1 routing**, **assisted top-3 suggestions**, or **not ready**.

All support content is synthetic and is not Indiana University guidance.

## What you are given

- this starter `.ipynb` notebook;
- a mock dataset containing 30 support articles, 10 evaluation questions, and 5 challenge questions;
- relevance labels for evaluating retrieval;
- a complete TF-IDF baseline and evaluation helpers; and
- validated fallback rankings for a temporary model-download outage.

The dataset is embedded in the notebook, so you do not need to find, scrape, upload, or connect to any external knowledge base.

## Expected time

This homework is designed for **2-3 hours maximum**. Most setup, data loading, TF-IDF, ranking, and metric code is provided. You complete two embedding calls and focus on interpreting evidence. If a genuine model-download problem would push you beyond three hours, use the automatic fallback and document the error in Checkpoint 2.

## Skills assessed

- distinguish lexical and dense representations;
- judge model-task fit from documentation;
- encode and rank queries and documents;
- evaluate top-k retrieval;
- investigate failures; and
- communicate a responsible recommendation.

## Submission

Submit one Google Colab notebook URL in Canvas. Share it as **Anyone with the link can view**, run all cells, leave outputs visible, and complete Graded Checkpoints 1-5. Do not submit a separate report or AI transcript.

## Responsible AI use

AI assistants may help with coding, debugging, documentation, or alternatives. You remain responsible for understanding and validating all code, outputs, and conclusions.


## 0. Environment setup

The setup installs the pinned Sentence Transformers version only when needed. A fresh Colab session normally needs internet access.

### Live mode and fallback mode

- **Live mode** means the notebook successfully loads the pinned Sentence Transformer and uses the two embedding calls you complete. The notebook creates new document and query embeddings, calculates similarity scores, and builds the rankings during your run. This is the expected mode.
- **Fallback mode** means the notebook uses instructor-provided top-three rankings that were previously generated and checked with the same pinned model. The fixed evaluation and challenge questions still work, but no new embeddings are created during your run.

You do not switch modes manually. Run the notebook and read the printed `Embedding retrieval mode`. Before you complete the two TODOs, the starter uses fallback mode. After correct implementation, it changes to live mode. If model loading genuinely fails, the fallback activates automatically; copy the reported error into Checkpoint 2 and continue with the interpretation tasks.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess, sys

REQUIRED_ST_VERSION = "5.1.0"
try:
    installed_st_version = version("sentence-transformers")
except PackageNotFoundError:
    installed_st_version = None

if installed_st_version != REQUIRED_ST_VERSION:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"sentence-transformers=={REQUIRED_ST_VERSION}",
    ])
else:
    print("sentence-transformers", installed_st_version, "is ready")


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)
np.set_printoptions(precision=4, suppress=True)


## 1. Inspect the provided mock dataset

The dataset below is already loaded from the starter notebook: 30 short support articles, 10 evaluation questions, and 5 challenge questions. Relevance labels are visible because the assignment assesses evaluation. Some questions reasonably have more than one acceptable article.


In [ ]:
ARTICLES_DATA = [{'doc_id': 'ACC-01', 'title': 'Reset a forgotten account password', 'intent_family': 'account_access', 'article_text': 'Use the Northstar account recovery page when you cannot remember your password. Enter your username, confirm a recovery email or phone number, and create a new password. This process changes the password used for email, course tools, wireless access, and other connected services. If you still know your password and only want to replace a sign-in approval device, use the device replacement article instead.'}, {'doc_id': 'ACC-02', 'title': 'Replace the phone used for sign-in approval', 'intent_family': 'account_access', 'article_text': 'Use the authentication device portal when you have a new phone or no longer have the device that received sign-in approvals. Choose Replace Device, verify your identity with a backup method, and enroll the new phone. Replacing an approval device does not require changing a password that still works. Contact the fictional help center if no backup method is available.'}, {'doc_id': 'ACC-03', 'title': 'Unlock an account after repeated sign-in failures', 'intent_family': 'account_access', 'article_text': 'An account may be temporarily locked after several unsuccessful sign-in attempts. Wait fifteen minutes, close saved login prompts, and try the correct password once. If the account remains locked, use the account recovery page or contact the fictional help center. A lockout is different from forgetting a username or replacing the phone used for sign-in approval.'}, {'doc_id': 'ACC-04', 'title': 'Change a password you still know', 'intent_family': 'account_access', 'article_text': 'Use account settings to change a password before it expires or whenever you want a new one. Sign in with the current password, enter the replacement twice, and update saved credentials on email, wireless, and mobile devices. If you cannot remember the current password, follow the forgotten password article. A password change does not replace an authentication phone.'}, {'doc_id': 'ACC-05', 'title': 'Find a forgotten university username', 'intent_family': 'account_access', 'article_text': 'Use the username lookup page if you remember your personal details but not the account name used to sign in. The lookup sends the username to a verified recovery address. This article does not reset a password, unlock an account, or change an authentication device. Contact the fictional help center if the recovery address is no longer available.'}, {'doc_id': 'NET-01', 'title': 'Connect a personal computer to secure campus Wi-Fi', 'intent_family': 'network_access', 'article_text': 'Choose Northstar Secure from the wireless menu on a personal laptop, enter the university username and password, and accept the fictional network certificate only when its name matches the setup page. Forget old campus networks if the computer repeatedly selects the wrong connection. Visitors without university accounts should use the guest network instructions instead.'}, {'doc_id': 'NET-02', 'title': 'Connect a visitor to the guest wireless network', 'intent_family': 'network_access', 'article_text': 'Visitors should select Northstar Guest and complete the browser registration page. A sponsor code may be required for extended access. University students and employees should use the secure campus wireless network instead because it provides access to more services. This article covers guest enrollment, not slow connections after a device is already online.'}, {'doc_id': 'NET-03', 'title': 'Troubleshoot slow or unstable wireless service', 'intent_family': 'network_access', 'article_text': 'When wireless service disconnects or becomes unusually slow, compare more than one website, move closer to an access point, disable and re-enable Wi-Fi, and test another device in the same location. Record the building, room, time, network name, and affected devices before reporting the problem. First-time connection failures should use the secure or guest setup articles.'}, {'doc_id': 'NET-04', 'title': 'Use a wired network connection in campus housing', 'intent_family': 'network_access', 'article_text': 'Connect an Ethernet cable from the room jack to the computer or approved adapter, then open a browser to complete device registration. Test another cable before reporting a dead wall jack. This procedure applies to wired residential connections. It does not configure secure Wi-Fi or register a screenless game console for wireless use.'}, {'doc_id': 'NET-05', 'title': 'Register a game console or screenless network device', 'intent_family': 'network_access', 'article_text': "Devices that cannot open a browser, such as some game consoles, streaming boxes, and smart devices, require manual network registration. Find the device's wireless hardware address, enter it in the Northstar device portal, and connect to the designated device network. Laptops and phones should use the normal secure wireless setup instead."}, {'doc_id': 'SFT-01', 'title': 'Install university-licensed software', 'intent_family': 'software_access', 'article_text': 'Open the Northstar software catalog to check whether a required application is available for your role and operating system. Review the license conditions, download the approved installer, and sign in when prompted. Some applications are available only through the virtual desktop. This article covers obtaining licensed software, not updating an application that is already installed.'}, {'doc_id': 'SFT-02', 'title': 'Connect to restricted services with the campus VPN', 'intent_family': 'software_access', 'article_text': 'Install the approved Northstar VPN client when an off-campus service requires a campus network connection. Open the client, choose the standard remote-access profile, sign in, and approve authentication. Disconnect when the restricted task is complete. The VPN is not normally needed for public websites, ordinary email, or the learning management system.'}, {'doc_id': 'SFT-03', 'title': 'Use the virtual desktop for remote applications', 'intent_family': 'software_access', 'article_text': 'The Northstar virtual desktop provides selected applications through a browser or remote desktop client. Sign in, choose an available desktop, and save important work to approved cloud storage before ending the session. Use this option when the software catalog says an application is virtual-only or when a personal computer cannot run the required program.'}, {'doc_id': 'SFT-04', 'title': 'Update an installed application', 'intent_family': 'software_access', 'article_text': "Check the application's About or Settings menu for its current version, save open work, and use the built-in update command or the approved installer from the software catalog. Restart the application after updating. If the program is not installed yet or the license is unavailable, use the licensed software article rather than this update procedure."}, {'doc_id': 'FIL-01', 'title': 'Resolve a cloud storage quota warning', 'intent_family': 'files_storage', 'article_text': 'A quota warning means the account is close to or above its allowed cloud storage. Sort files by size, remove unneeded duplicates, empty the deleted-items folder, and move archival material to an approved location. Deleting a file does not immediately reduce usage if it remains in the recycle area. This article does not address a missing file that needs recovery.'}, {'doc_id': 'FIL-02', 'title': 'Recover a recently deleted file', 'intent_family': 'files_storage', 'article_text': 'Open the cloud storage recycle area, locate the deleted item, and choose Restore. Also check version history if the file exists but recent content is missing. Recovery availability is limited, so act promptly and record the file name and approximate deletion time. Emptying the recycle area may make self-service recovery unavailable.'}, {'doc_id': 'FIL-03', 'title': 'Change access to a shared folder', 'intent_family': 'files_storage', 'article_text': "Open the folder's sharing settings to review who can view, comment, or edit. Change the permission for the intended person or group and remove broad public access that is no longer needed. Test the link in a private browser when possible. This article addresses permissions; it does not restore deleted files or resolve storage quota warnings."}, {'doc_id': 'FIL-04', 'title': 'Share or submit a file that is too large', 'intent_family': 'files_storage', 'article_text': 'When a file exceeds an upload or email limit, place it in approved cloud storage and share a view-only link with the intended audience. For course submissions, confirm that the instructor accepts links and that permission remains available through grading. Compress media only when quality requirements allow it. Do not split sensitive files across unapproved public services.'}, {'doc_id': 'PRN-01', 'title': 'Add a campus printer to a computer', 'intent_family': 'printing', 'article_text': 'Install the Northstar print client, choose the campus print queue, and follow the operating-system prompts to add it. Send a one-page test document before printing a large job. If a submitted job appears in the queue but no paper comes out, use the print release instructions. If the printer cannot be contacted, check the printer offline article.'}, {'doc_id': 'PRN-02', 'title': 'Check or add printing credit', 'intent_family': 'printing', 'article_text': 'Open the fictional print account page to review the remaining balance and recent charges. Add credit through the approved payment method if the balance is too low for the requested pages. A sufficient balance does not release a job waiting at a printer or repair an offline device. Report a disputed charge with the job time and printer location.'}, {'doc_id': 'PRN-03', 'title': 'Release a print job at the printer', 'intent_family': 'printing', 'article_text': 'After sending a document to the campus print queue, visit a release station, sign in, select the job, review the page count, and choose Print. Jobs expire after the fictional retention period. If the job is absent, confirm that the correct queue was selected. If it remains listed but the printer reports an error, also consult the printer offline article.'}, {'doc_id': 'PRN-04', 'title': 'Troubleshoot a printer that is offline or in error', 'intent_family': 'printing', 'article_text': 'Check the printer display for paper, toner, jam, or maintenance messages. Do not open secured panels. Try another nearby printer and report the device name, location, and displayed error. A job can also wait because it has not been released, so confirm its status in the print queue before assuming the printer itself is offline.'}, {'doc_id': 'SEC-01', 'title': 'Report a suspected phishing message', 'intent_family': 'security', 'article_text': "Do not open unexpected attachments or use a message link that asks for a password, payment, or urgent verification. Use the mail client's Report Phishing command or forward the message to the fictional security team as an attachment. If credentials were already entered, change the password from a trusted device and review the suspicious sign-in article."}, {'doc_id': 'SEC-02', 'title': 'Respond to suspected malware on a computer', 'intent_family': 'security', 'article_text': 'Disconnect the computer from networks if it displays unexpected encryption messages, repeated pop-ups, unknown software, or other signs of compromise. Do not continue entering passwords. Record what happened and contact the fictional security team from another device. A suspicious email alone should be reported with the phishing procedure instead of treated as confirmed malware.'}, {'doc_id': 'SEC-03', 'title': 'Report a lost or stolen laptop or phone', 'intent_family': 'security', 'article_text': 'Report a missing university-owned or personally enrolled device promptly. Provide the device type, asset tag when applicable, last known location, and approximate time it disappeared. Use an approved device-finding service only when it is safe to do so. Change exposed credentials and remove the device from sign-in approval if necessary. For a phone replacement without loss, use the authentication device article.'}, {'doc_id': 'SEC-04', 'title': 'Find an encryption recovery key', 'intent_family': 'security', 'article_text': 'An encrypted computer may request a recovery key after a hardware, firmware, or security change. Record the recovery-key identifier shown on screen and use the approved device-management portal to locate the matching key. Do not post the key in public messages. A recovery key unlocks encrypted storage; it is not the same as an account password reset.'}, {'doc_id': 'LRN-01', 'title': 'Troubleshoot a course assignment upload', 'intent_family': 'learning_tools', 'article_text': 'Confirm the assignment is open, the file type is allowed, and the upload finished before selecting Submit. Reopen the assignment to verify that a receipt or submitted file appears. If a video or dataset exceeds the stated limit, check whether the instructor permits an approved cloud-storage link. Preserve screenshots and the submission receipt when reporting a problem.'}, {'doc_id': 'LRN-02', 'title': 'Fix missing sound in a video meeting', 'intent_family': 'learning_tools', 'article_text': "In the meeting application, select the intended microphone and speaker, run the audio test, and check the operating system's privacy permissions. Disconnect unused headsets and close other programs that may control the microphone. If other websites also have no sound, test the device outside the meeting application before reporting the issue."}, {'doc_id': 'LRN-03', 'title': 'Connect a laptop to a classroom display', 'intent_family': 'learning_tools', 'article_text': "Select the room's presentation source, connect the labeled cable or approved wireless display, and choose Duplicate or Extend in the computer's display settings. Confirm that the correct adapter is attached and wake the room display before reconnecting. For sound, select the room audio device separately. Record the room number if the display remains blank."}, {'doc_id': 'LRN-04', 'title': 'Set up university email on a phone', 'intent_family': 'learning_tools', 'article_text': 'Add a new work or school account in the approved mail application, enter the university email address, complete the web sign-in, and approve authentication. Do not manually enter server settings unless the setup page specifically requires them. If mail was working but expected messages are missing, check filtering and quarantine rather than adding the account again.'}]

QUERIES_DATA = [{'query_id': 'Q01', 'split': 'evaluation', 'query_text': 'I got a new phone and cannot approve my university login anymore.', 'relevant_doc_ids': ['ACC-02'], 'challenge_type': 'semantic_paraphrase'}, {'query_id': 'Q02', 'split': 'evaluation', 'query_text': 'My laptop keeps dropping the wireless connection in the library.', 'relevant_doc_ids': ['NET-03'], 'challenge_type': 'symptom_description'}, {'query_id': 'Q03', 'split': 'evaluation', 'query_text': 'A campus visitor needs internet access for the afternoon.', 'relevant_doc_ids': ['NET-02'], 'challenge_type': 'low_lexical_overlap'}, {'query_id': 'Q04', 'split': 'evaluation', 'query_text': 'The statistics program required for my class is not installed on my computer.', 'relevant_doc_ids': ['SFT-01', 'SFT-03'], 'challenge_type': 'multiple_valid_documents'}, {'query_id': 'Q05', 'split': 'evaluation', 'query_text': 'I deleted a spreadsheet yesterday and need to get it back.', 'relevant_doc_ids': ['FIL-02'], 'challenge_type': 'semantic_paraphrase'}, {'query_id': 'Q06', 'split': 'evaluation', 'query_text': 'My project partner can open our folder but cannot make changes.', 'relevant_doc_ids': ['FIL-03'], 'challenge_type': 'implicit_permission_problem'}, {'query_id': 'Q07', 'split': 'evaluation', 'query_text': 'The document is in the campus print queue, but nothing has printed yet.', 'relevant_doc_ids': ['PRN-03', 'PRN-04'], 'challenge_type': 'ambiguous_state'}, {'query_id': 'Q08', 'split': 'evaluation', 'query_text': 'An email says my account will close unless I verify my password immediately.', 'relevant_doc_ids': ['SEC-01'], 'challenge_type': 'security_risk'}, {'query_id': 'Q09', 'split': 'evaluation', 'query_text': 'My presentation is ready, but the classroom screen stays blank.', 'relevant_doc_ids': ['LRN-03'], 'challenge_type': 'contextual_intent'}, {'query_id': 'Q10', 'split': 'evaluation', 'query_text': 'The course site rejects my video because the file is too big.', 'relevant_doc_ids': ['LRN-01', 'FIL-04'], 'challenge_type': 'cross_intent'}, {'query_id': 'C01', 'split': 'challenge', 'query_text': 'Do not reset my password. I can sign in, but I need approvals sent to my replacement phone.', 'relevant_doc_ids': ['ACC-02'], 'challenge_type': 'negation_and_lexical_trap'}, {'query_id': 'C02', 'split': 'challenge', 'query_text': 'The printer is online, and my job is waiting. Where do I make the pages come out?', 'relevant_doc_ids': ['PRN-03'], 'challenge_type': 'constraint_resolution'}, {'query_id': 'C03', 'split': 'challenge', 'query_text': 'How do I report a phising emial that asked for my pasword?', 'relevant_doc_ids': ['SEC-01'], 'challenge_type': 'misspelling'}, {'query_id': 'C04', 'split': 'challenge', 'query_text': 'A guest connected to Wi-Fi, but it becomes slow every few minutes.', 'relevant_doc_ids': ['NET-03'], 'challenge_type': 'distracting_role_term'}, {'query_id': 'C05', 'split': 'challenge', 'query_text': 'My phone was stolen, and it was also the device I used to approve logins.', 'relevant_doc_ids': ['SEC-03', 'ACC-02'], 'challenge_type': 'multi_intent'}]

FALLBACK_EMBEDDING_RANKINGS = {'Q01': [['ACC-02', 0.6845], ['LRN-04', 0.5557], ['ACC-03', 0.4603]], 'Q02': [['NET-03', 0.4631], ['LRN-03', 0.3535], ['NET-04', 0.3413]], 'Q03': [['SFT-02', 0.4976], ['NET-02', 0.4947], ['NET-01', 0.473]], 'Q04': [['SFT-04', 0.3279], ['SFT-01', 0.315], ['LRN-01', 0.215]], 'Q05': [['FIL-02', 0.4582], ['ACC-05', 0.3268], ['SEC-03', 0.3147]], 'Q06': [['FIL-03', 0.5011], ['SFT-04', 0.3859], ['LRN-01', 0.3043]], 'Q07': [['PRN-01', 0.5433], ['PRN-03', 0.4733], ['PRN-02', 0.3947]], 'Q08': [['ACC-03', 0.481], ['SEC-02', 0.4541], ['SEC-01', 0.42]], 'Q09': [['LRN-03', 0.3966], ['LRN-02', 0.3104], ['LRN-01', 0.2925]], 'Q10': [['LRN-01', 0.5279], ['FIL-04', 0.4506], ['FIL-03', 0.2745]], 'C01': [['ACC-02', 0.8176], ['ACC-04', 0.7199], ['ACC-03', 0.6096]], 'C02': [['PRN-01', 0.6323], ['PRN-03', 0.6128], ['PRN-04', 0.5612]], 'C03': [['SEC-01', 0.3888], ['SEC-02', 0.384], ['SEC-03', 0.223]], 'C04': [['NET-03', 0.5737], ['NET-02', 0.3981], ['NET-04', 0.3239]], 'C05': [['ACC-02', 0.5548], ['SEC-03', 0.5289], ['ACC-03', 0.4077]]}


articles = pd.DataFrame(ARTICLES_DATA)
queries = pd.DataFrame(QUERIES_DATA)
articles["document_text"] = articles["title"] + ". " + articles["article_text"]
evaluation_queries = queries.query("split == 'evaluation'").reset_index(drop=True)
challenge_queries = queries.query("split == 'challenge'").reset_index(drop=True)

print("Articles:", len(articles))
print("Evaluation queries:", len(evaluation_queries))
print("Challenge queries:", len(challenge_queries))
display(articles.groupby("intent_family", as_index=False).size().rename(columns={"size": "article_count"}))
display(articles[["doc_id", "title", "intent_family"]])


### Model and task fit

Read the [`multi-qa-MiniLM-L6-cos-v1` model card](https://huggingface.co/sentence-transformers/multi-qa-MiniLM-L6-cos-v1) and [semantic-search documentation](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html). The model card describes a 384-dimensional model designed for semantic search over questions and passages. This homework uses asymmetric retrieval: short questions are matched to longer answer passages.


## Graded Checkpoint 1: Retrieval brief — 15 points


Before viewing aggregate results, answer:

1. What user problem does the system solve?
2. Why is this asymmetric retrieval?
3. Why is the model plausible, and what is one documented limitation?
4. What top-1 and hit-at-3 results would justify automatic routing or assisted suggestions?

> Replace this line with your response.


## 2. TF-IDF baseline

Fit once on the article collection and transform every query in the same feature space. Do not refit on individual queries.


In [ ]:
def rankings_from_scores(score_matrix, query_frame, method, top_k=3):
    rows = []
    for qpos, qrow in query_frame.reset_index(drop=True).iterrows():
        relevant = set(qrow["relevant_doc_ids"])
        for rank, apos in enumerate(np.argsort(-score_matrix[qpos])[:top_k], start=1):
            article = articles.iloc[apos]
            rows.append({
                "query_id": qrow["query_id"], "split": qrow["split"],
                "method": method, "rank": rank, "doc_id": article["doc_id"],
                "title": article["title"], "score": float(score_matrix[qpos, apos]),
                "relevant": article["doc_id"] in relevant,
            })
    return pd.DataFrame(rows)

def fallback_to_frame(query_frame):
    rows = []
    titles = articles.set_index("doc_id")["title"]
    for _, qrow in query_frame.iterrows():
        relevant = set(qrow["relevant_doc_ids"])
        for rank, (doc_id, score) in enumerate(FALLBACK_EMBEDDING_RANKINGS[qrow["query_id"]], start=1):
            rows.append({
                "query_id": qrow["query_id"], "split": qrow["split"],
                "method": "Sentence Transformer", "rank": rank, "doc_id": doc_id,
                "title": titles.loc[doc_id], "score": float(score),
                "relevant": doc_id in relevant,
            })
    return pd.DataFrame(rows)

tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_documents = tfidf_vectorizer.fit_transform(articles["document_text"])
tfidf_queries = tfidf_vectorizer.transform(queries["query_text"])
tfidf_rankings = rankings_from_scores(
    cosine_similarity(tfidf_queries, tfidf_documents), queries, "TF-IDF"
)
print("TF-IDF document matrix:", tfidf_documents.shape)
display(tfidf_rankings.query("query_id == 'Q01'"))


## 3. Sentence Transformer retrieval

The model is pinned to a tested Hugging Face revision. The loading cell reports an error and enables fallback mode if the model cannot be obtained.


In [ ]:
MODEL_ID = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
MODEL_REVISION = "b207367332321f8e44f96e224ef15bc607f4dbf0"
model = None
model_load_error = None
try:
    model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)
    print("Loaded", MODEL_ID, "at", MODEL_REVISION)
except Exception as exc:
    model_load_error = f"{type(exc).__name__}: {exc}"
    print("Model unavailable; fixed-query fallback is enabled.")
    print(model_load_error)


In [ ]:
if model is not None:
    # TODO: Replace both placeholders with model.encode_document(...) and model.encode_query(...).
    # Request normalized NumPy embeddings. See the preceding model-card links.
    document_embeddings = None
    query_embeddings = None
else:
    # Keep these values as None so a genuine model-loading outage activates fallback mode.
    document_embeddings = query_embeddings = None

embedding_implementation_complete = (
    document_embeddings is not None and query_embeddings is not None
    and document_embeddings.shape == (30, 384) and query_embeddings.shape == (15, 384)
)
print("Live embedding implementation complete:", embedding_implementation_complete)


In [ ]:
if embedding_implementation_complete:
    embedding_rankings = rankings_from_scores(
        cosine_similarity(query_embeddings, document_embeddings), queries, "Sentence Transformer"
    )
    retrieval_mode = "live Sentence Transformer"
else:
    embedding_rankings = fallback_to_frame(queries)
    retrieval_mode = "validated fixed-query fallback"

print("Embedding retrieval mode:", retrieval_mode)
display(embedding_rankings.query("query_id == 'Q01'"))


## Graded Checkpoint 2: Build and compare retrieval — 30 points


In [ ]:
required_query_ids = ["Q02", "Q08", "Q10"]
comparison_evidence = pd.concat([tfidf_rankings, embedding_rankings], ignore_index=True)
comparison_evidence = comparison_evidence[comparison_evidence["query_id"].isin(required_query_ids)]
comparison_evidence["relevant_doc_ids"] = comparison_evidence["query_id"].map(
    queries.set_index("query_id")["relevant_doc_ids"]
)
display(comparison_evidence[[
    "query_id", "relevant_doc_ids", "method", "rank", "doc_id", "title", "score", "relevant"
]].sort_values(["query_id", "method", "rank"]))


Using Q02, Q08, and Q10:

1. State whether the printed retrieval mode was live or fallback. If fallback was caused by a genuine model-loading outage, include the reported error.
2. Cite IDs, ranks, and a score for one material representation difference.
3. Identify one query where both methods were useful.
4. Judge whether the selected articles actually address each user's need.

> Replace this line with your response.


## 4. Evaluate retrieval quality

Top-1 accuracy asks whether the first result is acceptable. Hit-at-3 asks whether at least one acceptable result appears among the first three.


In [ ]:
def evaluate(rankings, query_frame):
    rows = []
    labels = query_frame.set_index("query_id")["relevant_doc_ids"]
    for method, method_rows in rankings.groupby("method"):
        top1, hit3 = [], []
        for query_id, query_rows in method_rows.groupby("query_id"):
            retrieved = query_rows.sort_values("rank")["doc_id"].tolist()
            relevant = set(labels.loc[query_id])
            top1.append(retrieved[0] in relevant)
            hit3.append(bool(relevant.intersection(retrieved[:3])))
        rows.append({"method": method, "query_count": len(top1),
                     "top1_accuracy": np.mean(top1), "hit_at_3": np.mean(hit3)})
    return pd.DataFrame(rows).sort_values("method").reset_index(drop=True)

evaluation_rankings = pd.concat([
    tfidf_rankings.query("split == 'evaluation'"),
    embedding_rankings.query("split == 'evaluation'"),
], ignore_index=True)
metrics_summary = evaluate(evaluation_rankings, evaluation_queries)
display(metrics_summary.style.format({"top1_accuracy": "{:.1%}", "hit_at_3": "{:.1%}"}))

outcomes = []
for query_id in evaluation_queries["query_id"]:
    relevant = set(evaluation_queries.set_index("query_id").loc[query_id, "relevant_doc_ids"])
    row = {"query_id": query_id, "relevant_doc_ids": sorted(relevant)}
    for method, prefix in [("TF-IDF", "tfidf"), ("Sentence Transformer", "embedding")]:
        ids = evaluation_rankings.query("query_id == @query_id and method == @method").sort_values("rank")["doc_id"].tolist()
        row[f"{prefix}_top1"] = ids[0]
        row[f"{prefix}_top1_correct"] = ids[0] in relevant
        row[f"{prefix}_hit_at_3"] = bool(relevant.intersection(ids[:3]))
    outcomes.append(row)
query_outcomes = pd.DataFrame(outcomes)
display(query_outcomes)


## Graded Checkpoint 3: Evaluate retrieval quality — 25 points


Report both metrics for both methods. Then analyze one shared success, one embedding improvement, and one embedding failure or non-improvement. Cite query and document IDs. Explain why aggregate metrics alone do not establish safety or usefulness.

> Replace this line with your response.


## 5. Challenge the result

Choose one challenge. Inspect its query and type, then record a prediction before running retrieval.


In [ ]:
display(challenge_queries[["query_id", "query_text", "challenge_type"]])


### Prediction before execution

State your challenge ID. Predict one likely top result per method and explain the wording or constraint that may cause it.

> Replace this line with your response.


In [ ]:
CHALLENGE_QUERY_ID = "C01"  # Choose C01, C02, C03, C04, or C05.
if CHALLENGE_QUERY_ID not in set(challenge_queries["query_id"]):
    raise ValueError("Choose a challenge ID from C01 through C05.")
selected_challenge = challenge_queries.query("query_id == @CHALLENGE_QUERY_ID").iloc[0]
challenge_results = pd.concat([tfidf_rankings, embedding_rankings], ignore_index=True)
challenge_results = challenge_results.query("query_id == @CHALLENGE_QUERY_ID")
print("Query:", selected_challenge["query_text"])
print("Acceptable IDs:", selected_challenge["relevant_doc_ids"])
display(challenge_results[["method", "rank", "doc_id", "title", "score", "relevant"]]
        .sort_values(["method", "rank"]))


## Graded Checkpoint 4: Challenge and recommendation — 25 points


Compare the result with your prediction, explain the influential wording or constraint, judge each top result against the complete need, and state whether the test changes your interpretation. Propose one bounded improvement. Then write a recommendation of at most 200 words choosing automatic top-1, assisted top-3, or not ready. Cite two metrics, one failure, and one limitation.

> Replace this line with your response.


## Graded Checkpoint 5: AI Use and Validation — 5 points


In 3-5 sentences, report what you used an AI assistant for, one suggestion or output you accepted, and one thing you independently verified, changed, or rejected. If you did not use AI, state that and describe your validation.

> Replace this line with your response.


## 6. Transparent validation checks

These checks verify data integrity, feature alignment, ranking order, and metric ranges. They do not decide whether a result is useful.


In [ ]:
article_ids = set(articles["doc_id"])
assert len(articles) == 30 and articles["doc_id"].is_unique
assert len(evaluation_queries) == 10 and len(challenge_queries) == 5
assert queries["query_id"].is_unique
assert all(set(ids).issubset(article_ids) for ids in queries["relevant_doc_ids"])
assert tfidf_documents.shape[0] == 30
assert tfidf_documents.shape[1] == tfidf_queries.shape[1]

for ranking_frame in (tfidf_rankings, embedding_rankings):
    assert len(ranking_frame) == len(queries) * 3
    assert ranking_frame.groupby("query_id")["rank"].apply(list).map(lambda x: x == [1, 2, 3]).all()
    assert np.isfinite(ranking_frame["score"]).all()
    assert ranking_frame.groupby("query_id")["score"].apply(lambda x: x.is_monotonic_decreasing).all()

if embedding_implementation_complete:
    assert document_embeddings.shape == (30, 384)
    assert query_embeddings.shape == (15, 384)
    assert np.allclose(np.linalg.norm(document_embeddings, axis=1), 1.0, atol=1e-4)
    assert np.allclose(np.linalg.norm(query_embeddings, axis=1), 1.0, atol=1e-4)

assert set(metrics_summary["method"]) == {"TF-IDF", "Sentence Transformer"}
assert metrics_summary[["top1_accuracy", "hit_at_3"]].apply(lambda x: x.between(0, 1).all()).all()
print("All computational validation checks passed.")
print("Embedding retrieval mode:", retrieval_mode)
if not embedding_implementation_complete:
    print("Complete the live embedding cells or document a genuine model-service outage.")


## Final submission check

- [ ] All cells run in order.
- [ ] Graded Checkpoints 1-5 contain your own responses.
- [ ] Checkpoint 2 identifies live or fallback mode.
- [ ] If fallback was needed because of an outage, Checkpoint 2 includes the reported error.
- [ ] The Checkpoint 4 recommendation is no more than 200 words.
- [ ] Required outputs are visible without full matrices.
- [ ] The Colab link allows anyone with the link to view.
